# Test avec GradientBoostingRegressor

In [7]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

In [8]:
df = pd.read_csv("../data/gold.csv")

In [9]:
# column_titles = df.columns.tolist()
# column_titles

In [17]:
X = df.drop(["prix_median"], axis=1)
y = df['prix_median']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True, random_state=42)

categorial_features = ["proximité_autoroute"]

numerical_features = ['tx_crim', 'tx_residence','tx_commerce',
       'tx_nitriq', 'nb_piece', 'tx_ancienneté_parc_immo', 'distance_centre_emploi',
        'indice_impot_foncier',  'ratio_eleve_enseignant', 'tx_status_sociaux_eco_inf']

categorical_transformer = OneHotEncoder(sparse_output=True)
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorial_features),
        ('num', numerical_transformer, numerical_features)
    ],
    remainder="passthrough" 
)

# Créer un objet CatBoostRegressor
# cb_reg = GradientBoostingRegressor(random_state=42, verbose=False)
# cb_reg = GradientBoostingRegressor(n_estimators = 900, learning_rate = 0.09155068852731214, max_depth = 6, min_samples_split = 2, subsample = 0.8, loss = 'huber', criterion = 'squared_error', max_features = 1.0, random_state=42, verbose=False)
cb_reg = GradientBoostingRegressor(n_estimators = 600, learning_rate = 0.021825064275610876, max_depth = 7, min_samples_split = 6, subsample = 1.0, loss = 'huber', criterion = 'squared_error', max_features = 'sqrt', random_state=42, verbose=False)


# Créer un pipeline avec le préprocesseur et le modèle CatBoostRegressor
pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('cb_reg', cb_reg)
])

# Entraîner le pipeline sur les données d'entraînement
pipe.fit(X_train, y_train)
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

print("Mean squared error : ")
print("TRAIN :",mean_squared_error(y_train, y_pred_train))
print("TEST :",mean_squared_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("Mean absolute error : ")
print("TRAIN :",mean_absolute_error(y_train, y_pred_train))
print("TEST :",mean_absolute_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("R2 score : ")    
print("TRAIN :",r2_score(y_train, y_pred_train))
print("TEST :",r2_score(y_test, y_pred_test))

Mean squared error : 
TRAIN : 0.05549775840283384
TEST : 7.259684146135703
Mean absolute error : 
TRAIN : 0.12909850009156432
TEST : 1.7900912239549653
R2 score : 
TRAIN : 0.9993611651442881
TEST : 0.901004914334441


In [11]:
# with mlflow.start_run():
#     mlflow.log_param("model", best_model.named_steps['model'].__class__.__name__)
#     mlflow.log_param("n_estimators", best_model.named_steps['model'].n_estimators)
#     mlflow.log_param("max_depth", best_model.named_steps['model'].max_depth)
#     mlflow.log_metric("mae", mae)
#     mlflow.log_metric("rmse", rmse)
#     mlflow.log_metric("r2", r2)
#     mlflow.sklearn.log_model(best_model, "model")

### Optuna (recherche des hyperparametres)

In [12]:
import optuna
from sklearn.metrics import r2_score

In [13]:
def objective(trial):
    # Définition des hyperparamètres à optimiser
    n_estimators = trial.suggest_int('n_estimators', 100, 1000, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    subsample = trial.suggest_float('subsample', 0.1, 1.0, step=0.1)
    loss = trial.suggest_categorical('loss', ['absolute_error', 'squared_error', 'huber', 'quantile'])
    criterion = trial.suggest_categorical('criterion', ['squared_error', 'friedman_mse'])
    max_features = trial.suggest_categorical('max_features', [1.0, 'sqrt', 'log2'])
    
    # Création du modèle avec les hyperparamètres suggérés
    model = GradientBoostingRegressor(n_estimators=n_estimators,
                                      learning_rate=learning_rate,
                                      max_depth=max_depth,
                                      min_samples_split=min_samples_split,
                                      subsample=subsample,
                                      loss=loss,
                                      criterion=criterion,
                                      max_features=max_features,
                                      random_state=42)
    
    # Entraînement du modèle
    model.fit(X_train, y_train)
    
    # Prédiction sur les données de test
    y_pred = model.predict(X_test)
    
    # Calcul du score R2
    r2 = r2_score(y_test, y_pred)
    
    return r2


In [14]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2023-06-14 09:10:39,357] A new study created in memory with name: no-name-cbd083ef-e8c2-4142-9be2-d979b0493ff9
[I 2023-06-14 09:10:39,728] Trial 0 finished with value: 0.8729300012405135 and parameters: {'n_estimators': 500, 'learning_rate': 0.073116961812399, 'max_depth': 10, 'min_samples_split': 9, 'subsample': 0.2, 'loss': 'squared_error', 'criterion': 'friedman_mse', 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8729300012405135.
[I 2023-06-14 09:10:42,065] Trial 1 finished with value: 0.8701454709291397 and parameters: {'n_estimators': 1000, 'learning_rate': 0.057026484727563664, 'max_depth': 3, 'min_samples_split': 2, 'subsample': 0.2, 'loss': 'huber', 'criterion': 'friedman_mse', 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8729300012405135.
[I 2023-06-14 09:10:42,307] Trial 2 finished with value: 0.8824725635620972 and parameters: {'n_estimators': 400, 'learning_rate': 0.07335542016137657, 'max_depth': 4, 'min_samples_split': 3, 'subsample': 0.5, 'loss': '

In [15]:
best_params = study.best_params
best_value = study.best_value

In [16]:
print(best_params)
print(best_value)

{'n_estimators': 600, 'learning_rate': 0.021825064275610876, 'max_depth': 7, 'min_samples_split': 6, 'subsample': 1.0, 'loss': 'huber', 'criterion': 'squared_error', 'max_features': 'sqrt'}
0.926244267458906
